# 05 Coreset Validation
In Phase 5, we validate the data utility hypothesis. We compare our baseline MLP trained on a **random** 5-episode subset (10%) against the exact same MLP trained purely on the **brain-inspired coreset** (Episodes 8, 6, 32, 42, 10).

In [ ]:
import os
import sys
import json
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath('..'))

from src.baseline import run_baseline, BaselineConfig

# 1. Load the Random Baseline Metrics we generated in notebook 03
with open('../results/baseline_metrics.json', 'r', encoding='utf-8') as f:
    baseline_metrics = json.load(f)

print("Random Baseline Test MSE:", baseline_metrics['mse'])
print("Random Train Episodes:", baseline_metrics['train_episodes'])

## 1. Train on the Coreset

In [ ]:
coreset_episodes = [8, 6, 32, 42, 10]
feature_path = os.path.abspath('../results/features_resnet18.pt')
cfg = BaselineConfig(
    epochs=50,
    # Match the settings from Random Baseline exactly
)

print(f"Training on Brain-Inspired Coreset: {coreset_episodes} ...")
coreset_metrics = run_baseline(
    feature_path=feature_path,
    config=cfg,
    train_episodes=coreset_episodes
)

print("\n--- Results ---")
print("Coreset Test MSE:", coreset_metrics['mse'])

## 2. Compare Performance
We will visualize the Validation Loss convergence over epochs, and compare the final Per-Joint testing MSE.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Validation Loss Convergence
ax1.plot(baseline_metrics['val_losses'], label='Random Baseline (val)', color='gray', linestyle='--')
ax1.plot(coreset_metrics['val_losses'], label='Brain-Inspired Coreset (val)', color='blue')
ax1.set_title("Validation Loss Convergence")
ax1.set_xlabel("Epochs")
ax1.set_ylabel("MSE")
ax1.legend()

# Plot 2: Per-Joint Test Error
joints = list(range(7))
width = 0.35
ax2.bar([j - width/2 for j in joints], baseline_metrics['per_joint_mse'], width, label='Random Baseline', color='gray')
ax2.bar([j + width/2 for j in joints], coreset_metrics['per_joint_mse'], width, label='Coreset', color='blue')
ax2.set_title("Per-Joint Evaluation Error (Test Set)")
ax2.set_xlabel("Joint Index (0-6)")
ax2.set_ylabel("MSE")
ax2.legend()

plt.tight_layout()
plt.show()